[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/relevance_vector_machines.ipynb)

# Relevance Vector Machines: Bayesian Sparse Kernels

This notebook accompanies the blog post [Relevance Vector Machines: Bayesian Sparse Kernels](https://sesen.ai/blog/relevance-vector-machines-bayesian-sparse-kernels) (Bishop PRML chapter 7, post BP6).

We build an RVM from scratch in NumPy, fit it to the same sinusoidal regression data Bishop uses in figure 7.9, then extend to two-class classification on the moons dataset (Bishop figure 7.12). We compare both against scikit-learn's SVM.


## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

np.random.seed(0)


## 2. RVM from scratch

Two helper functions, then the regression class. The regression fit is the iterative re-estimation algorithm of Bishop equations 7.81-7.88: alternate between updating the Gaussian posterior `(m, Σ)` and updating the precisions `α` and `β` until convergence. Pruning happens implicitly: once `α_i` exceeds `1e9` we drop the corresponding basis function from the active set.


In [ ]:
ALPHA_PRUNE = 1e9


def rbf_kernel(X1, X2, gamma):
    sq = np.sum(X1 ** 2, axis=1)[:, None] + np.sum(X2 ** 2, axis=1)[None, :] - 2 * X1 @ X2.T
    return np.exp(-gamma * np.maximum(sq, 0.0))


def design_matrix(X_eval, X_basis, gamma):
    """Phi with a leading bias column."""
    K = rbf_kernel(X_eval, X_basis, gamma)
    bias = np.ones((X_eval.shape[0], 1))
    return np.hstack([bias, K])


class RVMRegressor:
    """RVM regression with an RBF basis centred on every training point."""

    def __init__(self, gamma=0.5, max_iter=200, tol=1e-3, alpha_prune=ALPHA_PRUNE):
        self.gamma = gamma
        self.max_iter = max_iter
        self.tol = tol
        self.alpha_prune = alpha_prune

    def fit(self, X, t):
        N = X.shape[0]
        Phi_full = design_matrix(X, X, self.gamma)
        M = Phi_full.shape[1]
        alpha = np.ones(M)
        beta = 1.0 / max(np.var(t) * 0.1, 1e-3)
        active = np.ones(M, dtype=bool)
        self.alpha_history_ = [alpha.copy()]
        self.active_history_ = [active.copy()]

        for _ in range(self.max_iter):
            Phi = Phi_full[:, active]
            a = alpha[active]
            Sigma = np.linalg.inv(np.diag(a) + beta * Phi.T @ Phi)
            m = beta * Sigma @ Phi.T @ t
            gamma_i = 1.0 - a * np.diag(Sigma)
            with np.errstate(divide="ignore", invalid="ignore"):
                a_new = np.where(m ** 2 > 1e-12, gamma_i / (m ** 2), self.alpha_prune * 10)
            beta_new = max(N - gamma_i.sum(), 1e-6) / max(np.sum((t - Phi @ m) ** 2), 1e-12)

            new_full = alpha.copy()
            new_full[active] = a_new
            change = np.max(np.abs(np.log(new_full[active] + 1e-12) - np.log(alpha[active] + 1e-12)))
            alpha = new_full
            beta = beta_new
            active = alpha < self.alpha_prune
            if active.sum() == 0:
                break
            self.alpha_history_.append(alpha.copy())
            self.active_history_.append(active.copy())
            if change < self.tol:
                break

        Phi = Phi_full[:, active]
        a = alpha[active]
        self.Sigma_ = np.linalg.inv(np.diag(a) + beta * Phi.T @ Phi)
        self.m_ = beta * self.Sigma_ @ Phi.T @ t
        self.alpha_, self.active_, self.beta_ = alpha, active, beta
        self.X_train_ = X
        active_idx = np.where(active)[0]
        self.relevance_vector_indices_ = active_idx[active_idx >= 1] - 1
        return self

    def predict(self, X_new, return_std=False):
        Phi_new = design_matrix(X_new, self.X_train_, self.gamma)[:, self.active_]
        mean = Phi_new @ self.m_
        if not return_std:
            return mean
        var = 1.0 / self.beta_ + np.sum((Phi_new @ self.Sigma_) * Phi_new, axis=1)
        return mean, np.sqrt(np.maximum(var, 0.0))


## 3. RVM regression on `sin(2πx)`

We fit the RVM and `scikit-learn`'s SVR to the same noisy sinusoid and count the surviving basis functions in each case.


In [ ]:
rng = np.random.default_rng(0)
xr = np.linspace(0, 1, 50).reshape(-1, 1)
tr = np.sin(2 * np.pi * xr.ravel()) + rng.normal(0, 0.1, xr.shape[0])

rvm = RVMRegressor(gamma=10.0).fit(xr, tr)
svr = SVR(kernel="rbf", gamma=10.0, C=10.0, epsilon=0.05).fit(xr, tr)

print(f"RVM: {rvm.relevance_vector_indices_.size} relevance vectors")
print(f"SVR: {len(svr.support_)} support vectors")


In [ ]:
xd = np.linspace(0, 1, 400).reshape(-1, 1)
mean, std = rvm.predict(xd, return_std=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(xd.ravel(), mean - std, mean + std, color="#dc2626", alpha=0.18, label=r"$\pm 1\sigma$")
ax.plot(xd, mean, color="#dc2626", lw=2, label="RVM mean")
ax.plot(xd, np.sin(2 * np.pi * xd.ravel()), color="#94a3b8", lw=1.2, ls="--", label=r"true $\sin(2\pi x)$")
ax.scatter(xr, tr, c="#22c55e", s=36, edgecolor="white", lw=1.0, label="training data", zorder=3)
rv = rvm.relevance_vector_indices_
ax.scatter(xr[rv], tr[rv], facecolors="none", edgecolors="#1d4ed8", s=170, lw=2.0,
           label=f"{rv.size} relevance vectors", zorder=4)
ax.set_xlabel("x"); ax.set_ylabel("t"); ax.set_xlim(0, 1)
ax.legend(loc="upper right", framealpha=0.95)
ax.grid(alpha=0.25)
plt.title(f"RVM regression: {rv.size} relevance vectors with predictive uncertainty")
plt.show()


## 4. Watch α diverge

The vast majority of basis-function precisions race past `1e9` within a few iterations; the survivors stabilise at small `α` and become the relevance vectors.


In [ ]:
hist = rvm.alpha_history_
active = rvm.active_history_
n = min(12, len(hist))

fig, ax = plt.subplots(figsize=(8, 4.5))
final_active = active[-1]
M = hist[0].size
for j in range(M):
    log_a = [np.log10(min(max(hist[k][j], 1e-3), 10 ** 9.5)) for k in range(n)]
    if final_active[j]:
        ax.plot(range(n), log_a, color="#dc2626", lw=2.2, alpha=0.95, zorder=5)
    else:
        ax.plot(range(n), log_a, color="#94a3b8", lw=0.7, alpha=0.55, zorder=2)
ax.axhline(9, color="#94a3b8", ls="--", lw=1.2, label=r"prune threshold $\log_{10} \alpha = 9$")
ax.set_xlabel("iteration"); ax.set_ylabel(r"$\log_{10} \alpha_i$")
ax.set_title("Most α diverge to ∞; the survivors are the relevance vectors")
ax.legend(loc="upper right"); ax.grid(alpha=0.25)
plt.show()


## 5. RVM for classification: Laplace plus IRLS

For binary classification we cannot integrate `w` analytically. Bishop (§7.2.3) uses a Laplace approximation: find the posterior mode by IRLS, then re-estimate `α` from the local Gaussian fit and iterate.


In [ ]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


class RVMClassifier:
    def __init__(self, gamma=1.5, max_iter=100, irls_iter=50, tol=1e-3, alpha_prune=ALPHA_PRUNE):
        self.gamma = gamma
        self.max_iter = max_iter
        self.irls_iter = irls_iter
        self.tol = tol
        self.alpha_prune = alpha_prune

    def _irls(self, Phi, t, alpha):
        w = np.zeros(Phi.shape[1])
        A = np.diag(alpha)
        for _ in range(self.irls_iter):
            y = _sigmoid(Phi @ w)
            B = y * (1.0 - y)
            g = Phi.T @ (t - y) - alpha * w
            H = -(Phi.T @ (B[:, None] * Phi) + A)
            try:
                step = np.linalg.solve(H, g)
            except np.linalg.LinAlgError:
                break
            w_new = w - step
            if np.max(np.abs(w_new - w)) < 1e-6:
                w = w_new
                break
            w = w_new
        Sigma = np.linalg.inv(Phi.T @ (B[:, None] * Phi) + A)
        return w, Sigma

    def fit(self, X, t):
        Phi_full = design_matrix(X, X, self.gamma)
        M = Phi_full.shape[1]
        alpha = np.ones(M)
        active = np.ones(M, dtype=bool)
        for _ in range(self.max_iter):
            Phi = Phi_full[:, active]
            a = alpha[active]
            w, Sigma = self._irls(Phi, t.astype(float), a)
            gamma_i = 1.0 - a * np.diag(Sigma)
            with np.errstate(divide="ignore", invalid="ignore"):
                a_new = np.where(w ** 2 > 1e-12, gamma_i / (w ** 2), self.alpha_prune * 10)
            new_full = alpha.copy()
            new_full[active] = a_new
            change = np.max(np.abs(np.log(new_full[active] + 1e-12) - np.log(alpha[active] + 1e-12)))
            alpha = new_full
            active = alpha < self.alpha_prune
            if active.sum() == 0:
                break
            if change < self.tol:
                break

        Phi = Phi_full[:, active]
        a = alpha[active]
        self.w_, self.Sigma_ = self._irls(Phi, t.astype(float), a)
        self.alpha_, self.active_ = alpha, active
        self.X_train_ = X
        active_idx = np.where(active)[0]
        self.relevance_vector_indices_ = active_idx[active_idx >= 1] - 1
        return self

    def predict_proba(self, X_new):
        Phi_new = design_matrix(X_new, self.X_train_, self.gamma)[:, self.active_]
        mu = Phi_new @ self.w_
        var = np.sum((Phi_new @ self.Sigma_) * Phi_new, axis=1)
        kappa = 1.0 / np.sqrt(1.0 + np.pi * var / 8.0)
        return _sigmoid(kappa * mu)

    def predict(self, X_new):
        return (self.predict_proba(X_new) >= 0.5).astype(int)


In [ ]:
Xc, yc = make_moons(n_samples=200, noise=0.22, random_state=3)
Xc = StandardScaler().fit_transform(Xc)

rvm_c = RVMClassifier(gamma=1.0).fit(Xc, yc)
svc = SVC(kernel="rbf", C=4.0, gamma=1.5, probability=True).fit(Xc, yc)

acc_rvm = ((rvm_c.predict_proba(Xc) >= 0.5) == yc).mean()
print(f"RVM: {rvm_c.relevance_vector_indices_.size} relevance vectors, train acc {acc_rvm:.3f}")
print(f"SVC: {len(svc.support_)} support vectors,    train acc {svc.score(Xc, yc):.3f}")


In [ ]:
gx = np.linspace(Xc[:, 0].min() - 0.3, Xc[:, 0].max() + 0.3, 200)
gy = np.linspace(Xc[:, 1].min() - 0.3, Xc[:, 1].max() + 0.3, 200)
GX, GY = np.meshgrid(gx, gy)
grid = np.c_[GX.ravel(), GY.ravel()]
P_rvm = rvm_c.predict_proba(grid).reshape(GX.shape)

fig, ax = plt.subplots(figsize=(7.5, 5))
cf = ax.contourf(GX, GY, P_rvm, levels=20, cmap="RdBu_r", vmin=0, vmax=1)
ax.contour(GX, GY, P_rvm, levels=[0.5], colors="black", linewidths=1.4)
ax.scatter(Xc[yc == 0, 0], Xc[yc == 0, 1], c="#1d4ed8", s=18, edgecolor="white", lw=0.4)
ax.scatter(Xc[yc == 1, 0], Xc[yc == 1, 1], c="#dc2626", s=18, edgecolor="white", lw=0.4)
rv_idx = rvm_c.relevance_vector_indices_
ax.scatter(Xc[rv_idx, 0], Xc[rv_idx, 1], facecolors="none", edgecolors="#22c55e",
           s=140, lw=1.8, label=f"{rv_idx.size} relevance vectors")
plt.colorbar(cf, ax=ax, label=r"$p(t = 1 \mid x)$")
ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
ax.set_title("RVM posterior probability map on the moons dataset")
ax.legend(loc="lower right", fontsize=9)
plt.show()


## 6. Exercises

1. **Median heuristic for `gamma`.** Replace the hard-coded `gamma=10.0` in the regression fit with `1.0 / np.median(pairwise distances)^2`. Does the resulting model still pick a sparse set of relevance vectors? Try the same for the classification dataset.

2. **Compare to a Gaussian process.** Use `sklearn.gaussian_process.GaussianProcessRegressor` with the same RBF kernel on the sinusoid. The GP keeps all 50 basis functions but produces wider predictive bands far from the training domain. Plot the RVM and GP standard deviations side-by-side and explain the difference.

3. **Multi-modal regression.** Generate `t = sin(2πx) + 0.5·sin(8πx) + noise`. Does the RVM still need only a handful of relevance vectors? At what `gamma` does it start to underfit?

4. **Adversarial extrapolation.** Predict at `x = 2.0` (well outside the training range `[0, 1]`). Compare the RVM predictive variance there to the GP's. Bishop's caveat about overconfident extrapolation should bite.

5. **Sparse Bayesian learning beyond kernels.** Apply the RVM machinery to a polynomial basis instead of an RBF kernel. Which basis functions does ARD prune first, and why?
